# Modèle 1 — Principal Component Regression (PCR)
**Projet ML 2026 — UMONS | Groupe 3**

La PCR combine deux étapes :
1. **PCA** (Principal Component Analysis) → réduit la dimensionnalité des features
2. **Régression linéaire** → prédit `Ja in Prozent` sur les composantes principales

C'est particulièrement utile ici car nos features sont nombreuses (67) et potentiellement corrélées entre elles.

In [ ]:
# Imports et configuration

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error

pd.set_option('display.float_format', '{:.4f}'.format)

# Chargement des données
train = pd.read_csv("../data/train_final.csv")
test  = pd.read_csv("../data/test_final.csv")

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")

test.head()

In [ ]:
# Séparation features / cible
y = train["Ja in Prozent"]

# One Hot Encoding sur Kanton
train_encoded = pd.get_dummies(train, columns=["Kanton"])
test_encoded  = pd.get_dummies(test,  columns=["Kanton"])

# Supprimer les colonnes inutiles
X = train_encoded.drop(columns=["Ja in Prozent", "Gemeinde", "commune_id"])
X_test = test_encoded.drop(columns=["Gemeinde", "commune_id"])

# Aligner les colonnes train et test (cantons manquants dans test → 0)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print(f"X train : {X.shape}")
print(f"X test  : {X_test.shape}")
print(f"y train : {y.shape}")
print(f"\nColonnes Kanton créées :")
print([col for col in X.columns if "Kanton" in col])

## Modèle PCR

In [ ]:
# Définition du pipeline PCR
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10)),
    ("regression", LinearRegression())
])

# Cross-validation 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipeline, X, y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse_scores = -scores
print("=== Résultats Cross-Validation (5 folds) ===")
print(f"RMSE par fold : {rmse_scores.round(4)}")
print(f"RMSE moyen    : {rmse_scores.mean():.4f}")
print(f"RMSE std      : {rmse_scores.std():.4f}")

## Optimisation du nombre de composantes
On teste différentes valeurs de n_components pour trouver le meilleur RMSE.

In [ ]:
# Optimisation du nombre de composantes principales
n_components_range = range(1, min(X.shape[0], X.shape[1]) + 1)
rmse_means = []
rmse_stds = []

for n in n_components_range:
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n)),
        ("regression", LinearRegression())
    ])
    scores = cross_val_score(
        pipeline, X, y,
        cv=kf,
        scoring="neg_root_mean_squared_error"
    )
    rmse_means.append(-scores.mean())
    rmse_stds.append(scores.std())

# Meilleur n_components
best_n = n_components_range[np.argmin(rmse_means)]
best_rmse = min(rmse_means)
print(f"Meilleur n_components : {best_n}")
print(f"Meilleur RMSE         : {best_rmse:.4f}")

# Visualisation
plt.figure(figsize=(10, 5))
plt.plot(n_components_range, rmse_means, color="steelblue", label="RMSE moyen")
plt.axvline(best_n, color="red", linestyle="--", label=f"Meilleur n={best_n}")
plt.xlabel("Nombre de composantes principales")
plt.ylabel("RMSE moyen (5 folds)")
plt.title("Optimisation du nombre de composantes PCR")
plt.legend()
plt.tight_layout()
plt.savefig("../figures/pcr_n_components.png", dpi=150, bbox_inches="tight")
plt.show()

##  Entraînement du modèle final
On entraîne le modèle PCR avec le meilleur n_components trouvé.

In [ ]:
# Entraînement du modèle final avec le meilleur n_components
pipeline_final = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=best_n)),
    ("regression", LinearRegression())
])

pipeline_final.fit(X, y)

# Evaluation finale sur le train complet
y_pred_train = pipeline_final.predict(X)
rmse_train = np.sqrt(mean_squared_error(y, y_pred_train))

print(f" Modèle PCR entraîné avec n_components={best_n}")
print(f"RMSE sur train complet : {rmse_train:.4f}")
print(f"RMSE cross-validation  : {best_rmse:.4f}")

## Prédictions et soumission Kaggle

In [ ]:
# Prédictions sur le test
y_pred_test = pipeline_final.predict(X_test)

# Créer le fichier de soumission
submission = pd.DataFrame({
    "Id": test["commune_id"],
    "Predicted": y_pred_test
})

submission.to_csv("../submissions/submission_PCR_v1.csv", index=False)

print(f"Soumission créée : {submission.shape}")
print(submission.head())